In [ ]:
# Características que dependan del movimiento

import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Cargar tu CSV de entrenamiento original
df = pd.read_csv('..\dataset_features_temperatura.csv')

# 2. LA LISTA DE VARIABLES SEGURAS (Inmunes a la posición y reinicios del SDR)
# Prohibimos phase_unwrap_k_mean, phase_circular_mean y svd_sigma_1
variables_seguras = [
    'doppler_variance_energy',  # El rey del movimiento térmico
    'dH_dt_mean',               # Inestabilidad temporal del canal
    'phase_jump_m_var_abs',     # Varianza de los saltos de fase (no fase absoluta)
    'phase_jump_k_var_abs',
    'phase_jump_m_rms_abs',
    'phase_unwrap_m_slope_var',
    'phase_unwrap_k_slope_var',
    'svd_sigma_ratio',          # Ratio SVD (mucho más estable que la energía absoluta)
    'doppler_spread'
]

# Filtrar para asegurarnos de que estas columnas existen en tu CSV
variables_finales = [v for v in variables_seguras if v in df.columns]

X = df[variables_finales]
y = df['temperature']

# 3. Entrenar el Modelo Robusto
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_robusto = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_robusto.fit(X_train, y_train)

# Comprobar R^2 (bajará un poco, pero será real)
y_pred = rf_robusto.predict(X_test)
print(f"R^2 del Modelo Robusto: {r2_score(y_test, y_pred):.4f}")

# 4. Sobrescribir el modelo guardado
joblib.dump(rf_robusto, 'modelo_RF_TOP10_Movimiento.pkl')
joblib.dump(variables_finales, 'nombres_variables_top10.pkl')
print("✅ Nuevo modelo robusto guardado con éxito. ¡Listo para probar en vivo!")

R^2 del Modelo Robusto: 0.8344
✅ Nuevo modelo robusto guardado con éxito. ¡Listo para probar en vivo!


In [3]:
# Prueba de modelo con características que dependen del movimiento

import sys
import pandas as pd
import joblib
from pathlib import Path

# =============================================================================
# 1. CONECTAR TU CARPETA CON LA DE HUGO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
    print("✅ Funciones de Hugo conectadas con éxito.")
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. CARGAR TU MODELO Y EL YAML
# =============================================================================
try:
    modelo_rf = joblib.load('modelo_RF_TOP10_Movimiento.pkl')
    variables_top10 = joblib.load('nombres_variables_top10.pkl')
    print("✅ Modelo de IA cargado.")
except FileNotFoundError:
    print("❌ ERROR: No encuentro el .pkl en tu carpeta.")
    exit()

# El YAML sigue estando en la carpeta 'data' general
RUTA_YAML = Path("../../data/Modulator.yaml")

# =============================================================================
# 3. RUTAS A TUS DATOS DE TEST (Directamente en tu carpeta)
# =============================================================================
# Como la carpeta está al lado de tu script, ya no hay que poner "../"
BBDD_DIR = Path("../Datos TEST Vaso carton")

# Lista con todas las muestras que tienes en tus subcarpetas
pruebas = [
    {
        "nombre": "Vaso Frío - Muestra 1",
        "tx": BBDD_DIR / "frio" / "iq_tx_1.bin",
        "rx": BBDD_DIR / "frio" / "iq_rx_1.bin"
    },
    {
        "nombre": "Vaso Frío - Muestra 2",
        "tx": BBDD_DIR / "frio" / "iq_tx_2.bin",
        "rx": BBDD_DIR / "frio" / "iq_rx_2.bin"
    },
    {
        "nombre": "Vaso Templado - Muestra 1",
        "tx": BBDD_DIR / "templado" / "iq_tx_1.bin", 
        "rx": BBDD_DIR / "templado" / "iq_rx_1.bin"
    },
    {
        "nombre": "Vaso Templado - Muestra 2",
        "tx": BBDD_DIR / "templado" / "iq_tx_2.bin", 
        "rx": BBDD_DIR / "templado" / "iq_rx_2.bin"
    },
    {
        "nombre": "Vaso Caliente - Muestra 1",
        "tx": BBDD_DIR / "caliente" / "iq_tx_1.bin",
        "rx": BBDD_DIR / "caliente" / "iq_rx_1.bin"
    },
    {
        "nombre": "Vaso Caliente - Muestra 2",
        "tx": BBDD_DIR / "caliente" / "iq_tx_2.bin",
        "rx": BBDD_DIR / "caliente" / "iq_rx_2.bin"
    }
]

# =============================================================================
# 4. LA PRUEBA FINAL
# =============================================================================
print("-" * 50)
print("INICIANDO PRUEBA DE INFERENCIA EN VIVO")
print("-" * 50)

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        print(f"⚠️ Aviso: Saltando '{p['nombre']}'. No encuentro sus archivos")
        continue
        
    try:
        # A. Estimar Canal (¡CON LOS PARÁMETROS EXACTOS DE HUGO!)
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], 
            rx_path=p["rx"], 
            yaml_path=RUTA_YAML, 
            n_symbols=None,
            start_sample_tx=0,
            start_sample_rx=0,
            storage_format_tx="auto",
            storage_format_rx="auto",
            fftshift=False,                # SUPER IMPORTANTE
            normalize_fft=False,           # SUPER IMPORTANTE
            trim_to_complete_frames=True,  # SUPER IMPORTANTE
            return_aux=False,
            output_order="mk", 
            verbose=False
        )
        
        # B. Extraer características
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # C. Filtrar (Tu Top 10)
        features_filtradas = {var: todas_features[var] for var in variables_top10}
        
        # D. Convertir a tabla asegurando que el orden de las columnas es perfecto
        df_pred = pd.DataFrame([features_filtradas], columns=variables_top10)
        
        # CHIVATO: Imprimimos el nombre de las variables y sus valores
        if "Muestra 1" in p["nombre"]:
            print(f"\n--- {p['nombre']} ---")
            for nombre_var, valor in list(features_filtradas.items())[:5]: # Muestra las 5 primeras
                print(f"  {nombre_var}: {valor}")
                
        # E. Predecir
        temp_detectada = modelo_rf.predict(df_pred)[0]
        
        print(f"🌡️ [{p['nombre']}] -> Predicción: {temp_detectada:.1f} ºC\n")
        
    except Exception as e:
        print(f"❌ Error procesando {p['nombre']}: {e}\n")

print("-" * 50)

✅ Funciones de Hugo conectadas con éxito.
✅ Modelo de IA cargado.
--------------------------------------------------
INICIANDO PRUEBA DE INFERENCIA EN VIVO
--------------------------------------------------

--- Vaso Frío - Muestra 1 ---
  doppler_variance_energy: 2.3485257432943856
  dH_dt_mean: 0.0004728900945875318
  phase_jump_m_var_abs: 0.004337660773379206
  phase_jump_k_var_abs: 0.008842512028943194
  phase_jump_m_rms_abs: 0.09464109868343266
🌡️ [Vaso Frío - Muestra 1] -> Predicción: 57.4 ºC

🌡️ [Vaso Frío - Muestra 2] -> Predicción: 45.0 ºC


--- Vaso Templado - Muestra 1 ---
  doppler_variance_energy: 3.2108921166698927
  dH_dt_mean: 0.000641472448004365
  phase_jump_m_var_abs: 0.001220408255844466
  phase_jump_k_var_abs: 0.002871156668922466
  phase_jump_m_rms_abs: 0.05337450230629323
🌡️ [Vaso Templado - Muestra 1] -> Predicción: 57.4 ºC

🌡️ [Vaso Templado - Muestra 2] -> Predicción: 57.4 ºC


--- Vaso Caliente - Muestra 1 ---
  doppler_variance_energy: 7.423988230906028
  dH

In [4]:
import pandas as pd

# Cargar tu BBDD original (asegúrate de que la ruta es correcta)
df = pd.read_csv('..\dataset_features_temperatura.csv')

print("--- LO QUE LA IA ESTUDIÓ EN EL EXCEL ---")
print(f"Doppler Energía Máxima: {df['doppler_variance_energy'].max():.4f}")
print(f"Fase Media Mínima: {df['phase_unwrap_k_mean'].min():.4f}")

--- LO QUE LA IA ESTUDIÓ EN EL EXCEL ---
Doppler Energía Máxima: 9062.6319
Fase Media Mínima: -324.2105


In [6]:
# Usamos características que no dependen de medidas ENERGIA ni AMPLITUD
# solo Frecuencia (Centroides), Ratios y Fases relativas

import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Cargar BBDD original
df = pd.read_csv('..\dataset_features_temperatura.csv')

# 2. LA LISTA DE SUPERVIVENCIA (Variables Inmunes a la Ganancia SDR)
# Quitamos todas las 'energy', 'sigma_1', 'var' que dependan de amplitud.
variables_invariantes = [
    'doppler_centroid',           # Es el centro de masa de la frecuencia (inmune a amplitud)
    'doppler_spread',             # Ancho de banda espectral (inmune a amplitud)
    'phase_circular_variance',    # Varianza en radianes (siempre de 0 a 1)
    'phase_jump_k_mean_abs',      # Saltos de fase promedio
    'phase_jump_m_mean_abs',
    'svd_sigma_ratio',            # SVD2 / SVD1 (Al dividir, la ganancia se anula matemáticamente)
    'dH_dt_phase_var'             # Si existe, varianza de fase temporal
]

# Nos quedamos solo con las que existan en el CSV de Hugo
variables_finales = [v for v in variables_invariantes if v in df.columns]

X = df[variables_finales]
y = df['temperature']

# 3. Entrenar el Modelo "Blindado"
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_blindado = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_blindado.fit(X_train, y_train)

print(f"R^2 del Modelo Invariante (BBDD antigua): {r2_score(y_test, rf_blindado.predict(X_test)):.4f}")

# 4. Guardar
joblib.dump(rf_blindado, 'modelo_RF_TOP10_Relativos.pkl') # Sobrescribimos el archivo
joblib.dump(variables_finales, 'nombres_variables_top10_relativos.pkl')
print(f"✅ Modelo guardado usando solo estas variables seguras:\n{variables_finales}")

R^2 del Modelo Invariante (BBDD antigua): 0.7575
✅ Modelo guardado usando solo estas variables seguras:
['doppler_centroid', 'doppler_spread', 'phase_circular_variance', 'phase_jump_k_mean_abs', 'phase_jump_m_mean_abs', 'svd_sigma_ratio']


In [7]:
# Prueba de modelo con características anterior

import sys
import pandas as pd
import joblib
from pathlib import Path

# =============================================================================
# 1. CONECTAR TU CARPETA CON LA DE HUGO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
    print("✅ Funciones de Hugo conectadas con éxito.")
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. CARGAR TU MODELO Y EL YAML
# =============================================================================
try:
    modelo_rf = joblib.load('modelo_RF_TOP10_Relativos.pkl')
    variables_top10 = joblib.load('nombres_variables_top10_relativos.pkl')
    print("✅ Modelo de IA cargado.")
except FileNotFoundError:
    print("❌ ERROR: No encuentro el .pkl en tu carpeta.")
    exit()

# El YAML sigue estando en la carpeta 'data' general
RUTA_YAML = Path("../../data/Modulator.yaml")

# =============================================================================
# 3. RUTAS A TUS DATOS DE TEST (Directamente en tu carpeta)
# =============================================================================
# Como la carpeta está al lado de tu script, ya no hay que poner "../"
BBDD_DIR = Path("../Datos TEST Vaso carton")

# Lista con todas las muestras que tienes en tus subcarpetas
pruebas = [
    {
        "nombre": "Vaso Frío - Muestra 1",
        "tx": BBDD_DIR / "frio" / "iq_tx_1.bin",
        "rx": BBDD_DIR / "frio" / "iq_rx_1.bin"
    },
    {
        "nombre": "Vaso Frío - Muestra 2",
        "tx": BBDD_DIR / "frio" / "iq_tx_2.bin",
        "rx": BBDD_DIR / "frio" / "iq_rx_2.bin"
    },
    {
        "nombre": "Vaso Templado - Muestra 1",
        "tx": BBDD_DIR / "templado" / "iq_tx_1.bin", 
        "rx": BBDD_DIR / "templado" / "iq_rx_1.bin"
    },
    {
        "nombre": "Vaso Templado - Muestra 2",
        "tx": BBDD_DIR / "templado" / "iq_tx_2.bin", 
        "rx": BBDD_DIR / "templado" / "iq_rx_2.bin"
    },
    {
        "nombre": "Vaso Caliente - Muestra 1",
        "tx": BBDD_DIR / "caliente" / "iq_tx_1.bin",
        "rx": BBDD_DIR / "caliente" / "iq_rx_1.bin"
    },
    {
        "nombre": "Vaso Caliente - Muestra 2",
        "tx": BBDD_DIR / "caliente" / "iq_tx_2.bin",
        "rx": BBDD_DIR / "caliente" / "iq_rx_2.bin"
    }
]

# =============================================================================
# 4. LA PRUEBA FINAL
# =============================================================================
print("-" * 50)
print("INICIANDO PRUEBA DE INFERENCIA EN VIVO")
print("-" * 50)

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        print(f"⚠️ Aviso: Saltando '{p['nombre']}'. No encuentro sus archivos")
        continue
        
    try:
        # A. Estimar Canal (¡CON LOS PARÁMETROS EXACTOS DE HUGO!)
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], 
            rx_path=p["rx"], 
            yaml_path=RUTA_YAML, 
            n_symbols=None,
            start_sample_tx=0,
            start_sample_rx=0,
            storage_format_tx="auto",
            storage_format_rx="auto",
            fftshift=False,                # SUPER IMPORTANTE
            normalize_fft=False,           # SUPER IMPORTANTE
            trim_to_complete_frames=True,  # SUPER IMPORTANTE
            return_aux=False,
            output_order="mk", 
            verbose=False
        )
        
        # B. Extraer características
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # C. Filtrar (Tu Top 10)
        features_filtradas = {var: todas_features[var] for var in variables_top10}
        
        # D. Convertir a tabla asegurando que el orden de las columnas es perfecto
        df_pred = pd.DataFrame([features_filtradas], columns=variables_top10)
        
        # CHIVATO: Imprimimos el nombre de las variables y sus valores
        if "Muestra 1" in p["nombre"]:
            print(f"\n--- {p['nombre']} ---")
            for nombre_var, valor in list(features_filtradas.items())[:5]: # Muestra las 5 primeras
                print(f"  {nombre_var}: {valor}")
                
        # E. Predecir
        temp_detectada = modelo_rf.predict(df_pred)[0]
        
        print(f"🌡️ [{p['nombre']}] -> Predicción: {temp_detectada:.1f} ºC\n")
        
    except Exception as e:
        print(f"❌ Error procesando {p['nombre']}: {e}\n")

print("-" * 50)

✅ Funciones de Hugo conectadas con éxito.
✅ Modelo de IA cargado.
--------------------------------------------------
INICIANDO PRUEBA DE INFERENCIA EN VIVO
--------------------------------------------------

--- Vaso Frío - Muestra 1 ---
  doppler_centroid: 0.25179745269951515
  doppler_spread: 0.28956403143620263
  phase_circular_variance: 0.9999024869416552
  phase_jump_k_mean_abs: 0.638175778996006
  phase_jump_m_mean_abs: 0.06796526161671147
🌡️ [Vaso Frío - Muestra 1] -> Predicción: 43.5 ºC

🌡️ [Vaso Frío - Muestra 2] -> Predicción: 42.8 ºC


--- Vaso Templado - Muestra 1 ---
  doppler_centroid: 0.25025601700269956
  doppler_spread: 0.2882627815266426
  phase_circular_variance: 0.9999410385676261
  phase_jump_k_mean_abs: 0.6381470294012881
  phase_jump_m_mean_abs: 0.04035380081974975
🌡️ [Vaso Templado - Muestra 1] -> Predicción: 43.5 ºC

🌡️ [Vaso Templado - Muestra 2] -> Predicción: 43.5 ºC


--- Vaso Caliente - Muestra 1 ---
  doppler_centroid: 0.25129278405930155
  doppler_spread

In [9]:
import pandas as pd

df = pd.read_csv('..\dataset_features_temperatura.csv')

print("--- CALIDAD DE SEÑAL: ENTRENAMIENTO VS TEST ---")
print(f"Varianza de Fase en BBDD Antigua (Media): {df['phase_circular_variance'].mean():.4f}")
print("Varianza de Fase en Prueba Cartón: 0.9999 (Ruido Total)")

--- CALIDAD DE SEÑAL: ENTRENAMIENTO VS TEST ---
Varianza de Fase en BBDD Antigua (Media): 0.9994
Varianza de Fase en Prueba Cartón: 0.9999 (Ruido Total)


Esos dos números que me acabas de dar son la pieza final del puzzle y explican absolutamente todo lo que nos ha pasado hoy.0.9994 en la BBDD Antigua vs 0.9999 en el Vaso de Cartón.Ambas señales son, a nivel de fase, puro ruido. Esto significa que la Inteligencia Artificial nos ha estado engañando desde el minuto uno. En Machine Learning, esto se conoce como el "Efecto de Hans el Listo" (Clever Hans Effect) o Fuga de Datos (Data Leakage).¿Qué es lo que ha pasado realmente en tu proyecto?La IA nunca aprendió a medir la temperatura: Al estar la señal ahogada en ruido (0.9994), el modelo Random Forest buscó a la desesperada cualquier número para poder aprobar el examen.La IA hizo trampas: Encontró que la variable doppler_variance_energy era gigantesca (9062). Esa variable no estaba midiendo el calor del agua, estaba midiendo cuánto tiempo duraba la grabación o a qué potencia estaba configurado el radar ese día concreto. Memorizó que "Grabación Tipo A = 15ºC" y sacó un $R^2$ de 0.97. Era un espejismo.El colapso en el mundo real: Al ponerle el vaso de cartón hoy, con un tiempo de grabación distinto o una ganancia diferente, la energía bajó a 2.34. Al quitarle las variables de energía para obligarle a mirar la forma de la onda, se encontró con que solo había ruido (0.9999). Como no había temperatura que rascar de ahí, colapsó a los 43 ºC.